# 0 — Preprocess

Flatten QBacMet JSON snapshots, merge with circuit metadata and QFw runtime data, and produce training datasets for MQBac.

**Outputs → `final_data/`**
- `qbacmet_flat.csv` — one row per QBacMet snapshot, all 6 layers flattened
- `best_backend_df.csv` — classification: best backend per config
- `estimate_runtime_df.csv` — regression: runtime for each backend

In [18]:
import os, json, glob, re
from pathlib import Path
import numpy as np
import pandas as pd

# ── Paths (edit if your layout differs) ──
BASE = Path("/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related")
STATS_DIRS = [BASE / "QBacMet" / "stats", BASE / "QBacMet" / "stats_sofar"]
APPS_GLOB  = str(BASE / "applications" / "*.csv")
RUNTIME_CSV  = BASE / "results_analysis" / "qfw_unified_summary.csv"
RUNTIME_JSON = BASE / "results_analysis" / "qfw_unified_data.json"
OUT_DIR = Path("final_data"); OUT_DIR.mkdir(exist_ok=True)

# Skip digit-prefixed test artifacts (0_nq20_..., 28_nq20_...)
SKIP_RE = re.compile(r"^\d+_nq")


## 1. Flatten QBacMet JSONs

In [19]:
def flatten_qbacmet_json(fp):
    """Flatten one QBacMet JSON into a single dict (one CSV row)."""
    with open(fp) as f:
        e = json.load(f)
    row = {"filename": Path(fp).name}

    # ── args ──
    args = e.get("args", {})
    for k, v in args.items():
        row[f"arg_{k}"] = v
    if "optimization_level" not in args:
        row["arg_optimization_level"] = (
            e.get("layers", {}).get("layer_2_transpile", {})
             .get("features", {}).get("optimization_level"))

    row["timestamp"] = e.get("timestamp")
    L = e.get("layers", {})

    # ── Layer 0: SLURM ──
    m0 = L.get("layer_0_slurm", {}).get("metrics", {})
    for k in ["num_nodes", "num_cpus", "num_tasks"]:
        row[f"slurm_{k}"] = m0.get(k)
    f0 = L.get("layer_0_slurm", {}).get("features", {})
    row["slurm_cluster"] = f0.get("frontend_cluster_name")
    row["slurm_account"] = f0.get("account")

    # ── Layer 1: Algorithm ──
    m1 = L.get("layer_1_algorithm", {}).get("metrics", {})
    for k in ["depth", "width", "num_qubits", "num_clbits", "num_ops",
              "num_2q_gates", "num_cliffords", "num_non_cliffords",
              "critical_path_length", "connected_components",
              "num_parameters", "shots"]:
        row[f"algo_{k}"] = m1.get(k)
    gc = m1.get("gate_counts", {})
    if isinstance(gc, dict):
        for g, cnt in gc.items():
            row[f"algo_gate_{g}"] = cnt

    # ── Layer 2: Transpile ──
    f2 = L.get("layer_2_transpile", {}).get("features", {})
    row["transpile_opt_level"] = f2.get("optimization_level")
    m2 = L.get("layer_2_transpile", {}).get("metrics", {})
    for k in ["hw_circ_depth", "hw_circ_width", "hw_circ_num_ops",
              "hw_circ_num_2q_gates", "hw_circ_swaps",
              "hw_circ_connected_components",
              "hw_circ_connectivity_degree_max",
              "hw_circ_connectivity_degree_avg"]:
        row[k] = m2.get(k)
    hgc = m2.get("hw_circ_gate_counts", {})
    if isinstance(hgc, dict):
        for g, cnt in hgc.items():
            row[f"hw_gate_{g}"] = cnt

    # ── Layer 3: Backend ──
    f3 = L.get("layer_3_backend", {}).get("features", {})
    for k in ["backend_name", "backend_type", "backend_is_simulator"]:
        row[f"be_{k}"] = f3.get(k)

    # ── Layer 4: Execution ──
    m4 = L.get("layer_4_execution", {}).get("metrics", {})
    for k in ["openmp_num_threads", "mpi_rank", "mpi_size", "transpile_time"]:
        row[f"exec_{k}"] = m4.get(k)

    # ── Derived ratios ──
    ad = row.get("algo_depth"); hd = row.get("hw_circ_depth")
    if ad and hd:
        try: row["depth_ratio"] = hd / ad
        except (TypeError, ZeroDivisionError): pass
    a2 = row.get("algo_num_2q_gates"); h2 = row.get("hw_circ_num_2q_gates")
    if a2 and h2:
        try: row["twoq_ratio"] = h2 / a2
        except (TypeError, ZeroDivisionError): pass

    return row


# ── Load all JSONs ──
rows = []
skipped = 0
for stats_dir in STATS_DIRS:
    if not stats_dir.exists():
        print(f"[warn] {stats_dir} not found"); continue
    for fp in sorted(stats_dir.glob("*.json")):
        if SKIP_RE.match(fp.name):
            skipped += 1; continue
        try:
            rows.append(flatten_qbacmet_json(fp))
        except Exception as ex:
            print(f"[err] {fp.name}: {ex}")

qbacmet_flat = pd.DataFrame(rows)

# ── Normalize benchmark names: nqmatrix* variants are HHL circuits ──
nq_mask = qbacmet_flat["arg_benchmark_name"].str.startswith("nqmatrix", na=False)
if nq_mask.any():
    print(f"Renaming {nq_mask.sum()} nqmatrix* rows → hhl")
    qbacmet_flat.loc[nq_mask, "arg_benchmark_name"] = "hhl"

print(f"Loaded {len(qbacmet_flat)} QBacMet snapshots ({skipped} test artifacts skipped)")
print(f"Columns: {qbacmet_flat.shape[1]}")

Renaming 15 nqmatrix* rows → hhl
Loaded 1222 QBacMet snapshots (58 test artifacts skipped)
Columns: 76


In [20]:
print("── Benchmarks ──")
print(qbacmet_flat["arg_benchmark_name"].value_counts().to_string())
print("\n── Backends ──")
print(qbacmet_flat["arg_simulator_type"].value_counts().to_string())
print("\n── Sub-backends ──")
print(qbacmet_flat["arg_sub_backend"].value_counts().to_string())
print("\n── Qubit counts ──")
print(sorted(qbacmet_flat["arg_number_of_qubits"].dropna().unique()))
print("\n── Opt levels ──")
print(qbacmet_flat["arg_optimization_level"].value_counts(dropna=False).to_string())
print(f"\n── Shape: {qbacmet_flat.shape} ──")
qbacmet_flat.head(3)


── Benchmarks ──
arg_benchmark_name
ghz            232
ham            208
tfim           182
mermin_bell    181
bit_code       174
phase_code     174
hhl             71

── Backends ──
arg_simulator_type
ionq       1028
ibmq        155
nwqsim       29
qtensor      10

── Sub-backends ──
arg_sub_backend
ideal                 175
aria-1                169
aria-2                168
forte-1               168
forte-enterprise-1    168
harmony               168
ibm_miami             114
ibm_torino             41
MPI                    29
simulator              12
numpy                  10

── Qubit counts ──
[np.int64(4), np.int64(5), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(15), np.int64(16), np.int64(17), np.int64(19), np.int64(20), np.int64(23), np.int64(29), np.int64(31), np.int64(39), np.int64(45)]

── Opt levels ──
arg_optimization_level
1.0    359
0.0    309
3.0    274
2.0    273
NaN      7

── Shape: (1222, 76) ──


,filename,arg_benchmark_name,arg_number_of_qubits,arg_simulator_type,arg_sub_backend,arg_device,arg_run_mode,arg_number_of_iterations,arg_optimization_level,timestamp,...,hw_gate_barrier,algo_gate_reset,hw_gate_reset,hw_gate_gpi2,hw_gate_ms,hw_gate_cx,hw_gate_gpi,hw_gate_ry,hw_gate_rx,hw_gate_h
0,ghz_nq15_ibmq_ibm_miami_CPU_sync_itrs1_opt0_20...,ghz,15,ibmq,ibm_miami,CPU,sync,1,0.0,2026-04-16T04:51:17.315483Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ghz_nq15_ibmq_ibm_miami_CPU_sync_itrs1_opt1_20...,ghz,15,ibmq,ibm_miami,CPU,sync,1,1.0,2026-04-16T04:51:17.317904Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ghz_nq15_ibmq_ibm_miami_CPU_sync_itrs1_opt2_20...,ghz,15,ibmq,ibm_miami,CPU,sync,1,2.0,2026-04-16T04:51:17.315192Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Load Circuit Metadata & QFw Runtime Data

In [21]:
# ── Circuit metadata from bench_meta_gen ──
csv_files = sorted(glob.glob(APPS_GLOB))
print(f"Found {len(csv_files)} circuit metadata CSVs")
if csv_files:
    benchmark_meta_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    benchmark_meta_df = benchmark_meta_df.drop_duplicates(subset=["benchmark", "n_qubits"])
    print(f"  benchmark_meta_df: {benchmark_meta_df.shape}")
else:
    raise FileNotFoundError(f"No CSVs at {APPS_GLOB}")

# ── QFw aggregated runtime (for classification — pick best backend by mean) ──
if RUNTIME_CSV.exists():
    backend_runtime_df = pd.read_csv(RUNTIME_CSV)
    print(f"  backend_runtime_df (aggregated): {backend_runtime_df.shape}")
else:
    raise FileNotFoundError(f"Runtime CSV not found: {RUNTIME_CSV}")

# ── QFw per-sample runtime (for regression — one row per timing measurement) ──
if RUNTIME_JSON.exists():
    with open(RUNTIME_JSON) as f:
        raw_data = json.load(f)

    persample_rows = []
    def _flatten_json(obj, keys=()):
        if isinstance(obj, list):
            bench, size, backend, sub_backend, device, run_mode, n_nodes, n_procs = keys
            for t_ms in obj:
                persample_rows.append({
                    "benchmark": bench, "size": int(size),
                    "backend": backend, "sub_backend": sub_backend,
                    "device": device, "run_mode": run_mode,
                    "n_nodes": int(n_nodes), "n_processes": int(n_procs),
                    "runtime_ms": t_ms,
                })
        elif isinstance(obj, dict):
            for k, v in obj.items():
                _flatten_json(v, keys + (k,))
    _flatten_json(raw_data)

    backend_runtime_persample_df = pd.DataFrame(persample_rows)
    print(f"  backend_runtime_persample_df (per-sample): {backend_runtime_persample_df.shape}")
    print(f"  Backends: {sorted(backend_runtime_persample_df['backend'].unique().tolist())}")
else:
    raise FileNotFoundError(f"Runtime JSON not found: {RUNTIME_JSON}")

display(benchmark_meta_df.head(3))
display(backend_runtime_persample_df.head(3))


Found 5 circuit metadata CSVs
  benchmark_meta_df: (84, 59)
  backend_runtime_df (aggregated): (524, 11)
  backend_runtime_persample_df (per-sample): (3220, 9)
  Backends: ['ibmq', 'ionq', 'nwqsim', 'qiskitaer', 'qtensor', 'tnqvm']


,benchmark,n_qubits,depth,width,n_clbits,n_ops,n_gates,n_parameters,n_connected_components,n_measure_ops,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,ghz,11,12,11,11,22,22,0,1,11,...,0,1,11,0,0,0,0,0,0,0
1,ghz,12,13,12,12,24,24,0,1,12,...,0,1,12,0,0,0,0,0,0,0
2,ghz,13,14,13,13,26,26,0,1,13,...,0,1,13,0,0,0,0,0,0,0


,benchmark,size,backend,sub_backend,device,run_mode,n_nodes,n_processes,runtime_ms
0,ghz,4,ibmq,ibm_miami,CPU,sync,1,8,104.019642
1,ghz,4,ibmq,ibm_miami,CPU,sync,1,8,33.597231
2,ghz,4,ibmq,ibm_miami,CPU,sync,1,8,28.068066


## 3. Build Classification Dataset

For each `(benchmark, size, n_nodes, n_processes)` → backend with lowest `mean_ms`.

In [22]:
group_cols = ["benchmark", "size", "n_nodes", "n_processes"]
idx = backend_runtime_df.groupby(group_cols)["mean_ms"].idxmin()
best_rows = backend_runtime_df.loc[
    idx, group_cols + ["backend", "sub_backend", "mean_ms"]
].copy()
best_rows = best_rows.rename(
    columns={"backend": "best_backend", "mean_ms": "best_runtime"}
)

# Merge with circuit metadata (size <-> n_qubits)
meta_for_join = benchmark_meta_df.copy()
meta_for_join["size"] = meta_for_join["n_qubits"].astype("Int64")
best_backend_df = pd.merge(
    best_rows, meta_for_join, how="left", on=["benchmark", "size"]
)

n_miss = best_backend_df["depth"].isna().sum()
print(f"best_backend_df: {best_backend_df.shape}")
if n_miss:
    print(f"  {n_miss} rows missing circuit metadata (DQAOA/QAOA size mismatch)")
best_backend_df.head(5)


best_backend_df: (215, 65)
  136 rows missing circuit metadata (DQAOA/QAOA size mismatch)


,benchmark,size,n_nodes,n_processes,best_backend,sub_backend,best_runtime,n_qubits,depth,width,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,bit_code,4,1,8,ibmq,ibm_torino,50.197601,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,bit_code,9,1,8,ibmq,ibm_torino,51.785946,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,bit_code,15,1,8,ibmq,ibm_torino,48.943520,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bit_code,16,1,8,ibmq,ibm_torino,53.647757,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,bit_code,20,1,8,ibmq,ibm_torino,56.928635,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Build Regression Dataset

All `(benchmark, size, backend, ...)` rows with individual `runtime_ms` as the target — one row per timing sample.


In [23]:
estimate_runtime_df = pd.merge(
    backend_runtime_persample_df, meta_for_join, how="left", on=["benchmark", "size"]
)
n_miss = estimate_runtime_df["depth"].isna().sum()
print(f"estimate_runtime_df: {estimate_runtime_df.shape}")
if n_miss:
    pct = 100 * n_miss / len(estimate_runtime_df)
    print(f"  {n_miss} rows ({pct:.1f}%) missing circuit metadata")
print(f"  Target column: runtime_ms")
estimate_runtime_df.head(5)


estimate_runtime_df: (3220, 67)
  1571 rows (48.8%) missing circuit metadata
  Target column: runtime_ms


,benchmark,size,backend,sub_backend,device,run_mode,n_nodes,n_processes,runtime_ms,n_qubits,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,ghz,4,ibmq,ibm_miami,CPU,sync,1,8,104.019642,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ghz,4,ibmq,ibm_miami,CPU,sync,1,8,33.597231,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ghz,4,ibmq,ibm_miami,CPU,sync,1,8,28.068066,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ghz,4,ibmq,ibm_miami,CPU,sync,1,8,151.770592,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ghz,4,ibmq,ibm_torino,CPU,sync,1,8,44.252872,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4b. Merge Cloud History (IBM / IonQ)

If `get_more_data.ipynb` has been run and saved `historic_runs_for_estimate_runtime_df.csv`, append those rows into `estimate_runtime_df` so all downstream modeling sees both HPC sim runs and real cloud runs.

In [24]:
# ── 4b.1: Legacy unified cloud history (from get_more_data.ipynb) ──
CLOUD_CSV = OUT_DIR / "historic_runs_for_estimate_runtime_df.csv"

if CLOUD_CSV.exists():
    cloud_df = pd.read_csv(CLOUD_CSV)
    core_cols = ["benchmark", "size", "backend", "sub_backend", "device",
                 "run_mode", "n_nodes", "n_processes", "runtime_ms"]
    extra_cols = [c for c in cloud_df.columns if c not in core_cols]
    cloud_slim = cloud_df[core_cols + extra_cols].copy()
    cloud_slim["runtime_ms"] = pd.to_numeric(cloud_slim["runtime_ms"], errors="coerce")
    cloud_slim = cloud_slim.dropna(subset=["runtime_ms"])
    estimate_runtime_df = pd.concat([estimate_runtime_df, cloud_slim],
                                    ignore_index=True, sort=False)
    print(f"Merged {len(cloud_slim)} unified-cloud rows → estimate_runtime_df now {estimate_runtime_df.shape}")
else:
    print(f"[info] No unified cloud history at {CLOUD_CSV} — skipping.")

# ── 4b.2: Per-collaborator <USER>_data.csv files (from get_more_data_ibmq.ipynb) ──
# Each collaborator runs `get_more_data_ibmq.ipynb` with their own USER_TAG and
# drops a `<USER>_data.csv` here. We glob all of them and append with the
# columns mapped to estimate_runtime_df's schema.
RESERVED = {CLOUD_CSV.name,
            "estimate_runtime_df.csv",
            "best_backend_df.csv",
            "qbacmet_flat.csv",
            "historic_runs_unified.csv"}

user_csvs = [p for p in sorted(OUT_DIR.glob("*_data.csv")) if p.name not in RESERVED]
print(f"\nFound {len(user_csvs)} per-user data CSVs in {OUT_DIR}/")

merged_rows = 0
for p in user_csvs:
    try:
        udf = pd.read_csv(p)
    except Exception as e:
        print(f"  [skip] {p.name}: {e}")
        continue
    if udf.empty or "runtime_ms" not in udf.columns:
        print(f"  [skip] {p.name}: no runtime_ms column")
        continue

    udf = udf.copy()
    udf["runtime_ms"] = pd.to_numeric(udf["runtime_ms"], errors="coerce")
    udf = udf.dropna(subset=["runtime_ms"])
    if udf.empty:
        print(f"  [skip] {p.name}: 0 rows with runtime_ms")
        continue

    # ── Schema mapping → estimate_runtime_df columns ──
    # provider (e.g. "ibmq")              → backend
    # backend (e.g. "ibm_torino")         → sub_backend
    # n_qubits                            → size  (cloud jobs lack a benchmark label)
    # n_nodes/n_processes                 → 1/1   (cloud is a single quantum job)
    # device                              → "QPU" (real hardware)
    # run_mode                            → "sync"
    # benchmark                           → "ibmq_cloud" (placeholder bucket)
    udf_out = pd.DataFrame({
        "benchmark":    udf.get("benchmark", "ibmq_cloud"),
        "size":         pd.to_numeric(udf.get("n_qubits"), errors="coerce"),
        "backend":      udf.get("provider", "ibmq"),
        "sub_backend":  udf.get("backend"),
        "device":       udf.get("device", "QPU"),
        "run_mode":     udf.get("run_mode", "sync"),
        "n_nodes":      pd.to_numeric(udf["n_nodes"] if "n_nodes" in udf.columns else pd.Series(1, index=udf.index), errors="coerce").fillna(1).astype(int),
        "n_processes":  pd.to_numeric(udf["n_processes"] if "n_processes" in udf.columns else pd.Series(1, index=udf.index), errors="coerce").fillna(1).astype(int),
        "runtime_ms":   udf["runtime_ms"],
    })

    # Carry over circuit-metric columns when present.
    for col in ["depth", "n_ops", "single_qubit_gates", "two_qubit_gates",
                "three_qubit_gates", "measure_ops", "critical_path_length",
                "connected_components", "num_cliffords", "num_non_cliffords",
                "num_parameters", "gate_counts_json", "circuit_metrics_json"]:
        if col in udf.columns:
            udf_out[col] = udf[col]

    # Need at least size to be useful (ties cloud rows to a circuit width).
    udf_out = udf_out.dropna(subset=["size"])
    udf_out["size"] = udf_out["size"].astype(int)

    estimate_runtime_df = pd.concat([estimate_runtime_df, udf_out],
                                    ignore_index=True, sort=False)
    merged_rows += len(udf_out)
    print(f"  [ok]  {p.name}: +{len(udf_out)} rows (sub_backends: {sorted(udf_out['sub_backend'].dropna().unique().tolist())})")

print(f"\nTotal per-user rows merged: {merged_rows}")
print(f"estimate_runtime_df now: {estimate_runtime_df.shape}")
print(f"Backends after all merges: {sorted(estimate_runtime_df['backend'].dropna().unique().tolist())}")

Merged 80 unified-cloud rows → estimate_runtime_df now (3300, 72)

Found 1 per-user data CSVs in final_data/
  [ok]  schundu3_data.csv: +200 rows (sub_backends: ['ibm_boston', 'ibm_fez', 'ibm_kingston', 'ibm_miami', 'ibm_pittsburgh', 'ibm_torino'])

Total per-user rows merged: 200
estimate_runtime_df now: (3500, 80)
Backends after all merges: ['ibmq', 'ionq', 'nwqsim', 'qiskitaer', 'qtensor', 'tnqvm']


## 5. Save & Summary

In [25]:
# Save CSVs
qbacmet_flat.to_csv(OUT_DIR / "qbacmet_flat.csv", index=False)
best_backend_df.to_csv(OUT_DIR / "best_backend_df.csv", index=False)
estimate_runtime_df.to_csv(OUT_DIR / "estimate_runtime_df.csv", index=False)

print("Saved to final_data/:")
for f in sorted(OUT_DIR.glob("*.csv")):
    n = len(pd.read_csv(f))
    print(f"  {f.name}: {n} rows")

print(f"\n── Numbers for the paper ──")
print(f"QBacMet snapshots: {len(qbacmet_flat)}")
print(f"Unique benchmarks: {qbacmet_flat['arg_benchmark_name'].nunique()}")
print(f"Unique sim types:  {qbacmet_flat['arg_simulator_type'].nunique()}")
print(f"Unique sub-backends: {qbacmet_flat['arg_sub_backend'].nunique()}")
print(f"Qubit range: {int(qbacmet_flat['arg_number_of_qubits'].min())}"
      f"–{int(qbacmet_flat['arg_number_of_qubits'].max())}")
print(f"Classification rows: {len(best_backend_df)}")
print(f"Regression rows:     {len(estimate_runtime_df)} ({backend_runtime_persample_df.shape[0]} individual timing samples)")


Saved to final_data/:
  best_backend_df.csv: 215 rows
  estimate_runtime_df.csv: 3500 rows
  historic_runs_for_estimate_runtime_df.csv: 80 rows
  historic_runs_unified.csv: 80 rows
  qbacmet_flat.csv: 1222 rows
  schundu3_data.csv: 200 rows

── Numbers for the paper ──
QBacMet snapshots: 1222
Unique benchmarks: 7
Unique sim types:  4
Unique sub-backends: 11
Qubit range: 4–45
Classification rows: 215
Regression rows:     3500 (3220 individual timing samples)
